<a href="https://colab.research.google.com/github/ClaraWongChiaCi/STQD6324_Assignment_1/blob/main/P1662442_STQD6324_Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [86]:
!pip install pyspark

In [87]:
from pyspark.sql import SparkSession

In [88]:
spark = SparkSession.builder.appName("Iris").getOrCreate()
spark

In [89]:
#Loading of Iris Dataset
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris()
iris

{'data': array([[5.1, 3.5, 1.4, 0.2],
        [4.9, 3. , 1.4, 0.2],
        [4.7, 3.2, 1.3, 0.2],
        [4.6, 3.1, 1.5, 0.2],
        [5. , 3.6, 1.4, 0.2],
        [5.4, 3.9, 1.7, 0.4],
        [4.6, 3.4, 1.4, 0.3],
        [5. , 3.4, 1.5, 0.2],
        [4.4, 2.9, 1.4, 0.2],
        [4.9, 3.1, 1.5, 0.1],
        [5.4, 3.7, 1.5, 0.2],
        [4.8, 3.4, 1.6, 0.2],
        [4.8, 3. , 1.4, 0.1],
        [4.3, 3. , 1.1, 0.1],
        [5.8, 4. , 1.2, 0.2],
        [5.7, 4.4, 1.5, 0.4],
        [5.4, 3.9, 1.3, 0.4],
        [5.1, 3.5, 1.4, 0.3],
        [5.7, 3.8, 1.7, 0.3],
        [5.1, 3.8, 1.5, 0.3],
        [5.4, 3.4, 1.7, 0.2],
        [5.1, 3.7, 1.5, 0.4],
        [4.6, 3.6, 1. , 0.2],
        [5.1, 3.3, 1.7, 0.5],
        [4.8, 3.4, 1.9, 0.2],
        [5. , 3. , 1.6, 0.2],
        [5. , 3.4, 1.6, 0.4],
        [5.2, 3.5, 1.5, 0.2],
        [5.2, 3.4, 1.4, 0.2],
        [4.7, 3.2, 1.6, 0.2],
        [4.8, 3.1, 1.6, 0.2],
        [5.4, 3.4, 1.5, 0.4],
        [5.2, 4.1, 1.5, 0.1],
  

In [90]:
df_iris = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df_iris.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [91]:
df_iris['label'] = iris.target
df_iris.tail()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),label
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2
149,5.9,3.0,5.1,1.8,2


In [92]:
#Convert to spark dataframe
spark_df = spark.createDataFrame(df_iris)
spark_df.show(5)

+-----------------+----------------+-----------------+----------------+-----+
|sepal length (cm)|sepal width (cm)|petal length (cm)|petal width (cm)|label|
+-----------------+----------------+-----------------+----------------+-----+
|              5.1|             3.5|              1.4|             0.2|    0|
|              4.9|             3.0|              1.4|             0.2|    0|
|              4.7|             3.2|              1.3|             0.2|    0|
|              4.6|             3.1|              1.5|             0.2|    0|
|              5.0|             3.6|              1.4|             0.2|    0|
+-----------------+----------------+-----------------+----------------+-----+
only showing top 5 rows


In [93]:
#The label column was cast to a Double type to ensure compatibility with Spark MLib
#classification algorithms and evaluators, which prefers floating points labels for mathematical processing
from pyspark.sql.functions import col
spark_df = spark_df.withColumn('label', col('label').cast('double'))

In [94]:
spark_df.printSchema()

root
 |-- sepal length (cm): double (nullable = true)
 |-- sepal width (cm): double (nullable = true)
 |-- petal length (cm): double (nullable = true)
 |-- petal width (cm): double (nullable = true)
 |-- label: double (nullable = true)



In [95]:
#To convert into a vector column using VectorAssembler whereby multiple feature columns are combined into a single vector column
#required by Spark MLlib for processing and compatibility across ML algorithms.
from pyspark.ml.feature import VectorAssembler

vector = VectorAssembler(inputCols=iris.feature_names, outputCol="features")
data = vector.transform(spark_df)
data = data.select("features", "label")

#To ensure randomSplit remains consistent across different model runs
data.cache()

data.show(5)

+-----------------+-----+
|         features|label|
+-----------------+-----+
|[5.1,3.5,1.4,0.2]|  0.0|
|[4.9,3.0,1.4,0.2]|  0.0|
|[4.7,3.2,1.3,0.2]|  0.0|
|[4.6,3.1,1.5,0.2]|  0.0|
|[5.0,3.6,1.4,0.2]|  0.0|
+-----------------+-----+
only showing top 5 rows


In [96]:
#Splitting of data into train & test sets
train_data, test_data = data.randomSplit([0.7, 0.3], seed=42)

In [97]:
#Logistic Regression
from pyspark.ml.classification import LogisticRegression
LR = LogisticRegression(featuresCol='features', labelCol='label')

In [98]:
#Hyperparameter tuning
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
paramGrid = (ParamGridBuilder()
              .addGrid(LR.regParam, [0.001, 0.01, 0.1, 1.0])
              .addGrid(LR.elasticNetParam, [0.0, 0.5, 1.0])
              .build())

In [99]:
#Cross-validation
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

cv_LR = CrossValidator(
    estimator=LR,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2 #Trains up to 2 models at once
)

In [100]:
#model fitting
LRmodel = cv_LR.fit(train_data)

In [101]:
#Extract parameters and average accuracy scores
LRparameters = LRmodel.getEstimatorParamMaps()
LRmetrics = LRmodel.avgMetrics

#zip together and sort to see the best performing combinations at the top
LRresults = sorted(zip(LRmetrics, LRparameters), key=lambda x: x[0], reverse=True)

print("Logistic Regression Hyperparameter Ranking (Best to Worst):")
for score, param_map in LRresults:
  param_details = {p.name: v for p, v in param_map.items()}
  print(f"Accuracy: {score:.4f} | Parameters: {param_details}")

Logistic Regression Hyperparameter Ranking (Best to Worst):
Accuracy: 0.9702 | Parameters: {'regParam': 0.01, 'elasticNetParam': 0.0}
Accuracy: 0.9702 | Parameters: {'regParam': 0.01, 'elasticNetParam': 0.5}
Accuracy: 0.9583 | Parameters: {'regParam': 0.01, 'elasticNetParam': 1.0}
Accuracy: 0.9487 | Parameters: {'regParam': 0.001, 'elasticNetParam': 0.0}
Accuracy: 0.9380 | Parameters: {'regParam': 0.001, 'elasticNetParam': 1.0}
Accuracy: 0.9368 | Parameters: {'regParam': 0.001, 'elasticNetParam': 0.5}
Accuracy: 0.9261 | Parameters: {'regParam': 0.1, 'elasticNetParam': 0.5}
Accuracy: 0.9130 | Parameters: {'regParam': 0.1, 'elasticNetParam': 1.0}
Accuracy: 0.8952 | Parameters: {'regParam': 0.1, 'elasticNetParam': 0.0}
Accuracy: 0.7998 | Parameters: {'regParam': 1.0, 'elasticNetParam': 0.0}
Accuracy: 0.4005 | Parameters: {'regParam': 1.0, 'elasticNetParam': 0.5}
Accuracy: 0.4005 | Parameters: {'regParam': 1.0, 'elasticNetParam': 1.0}


Hyperparameter tuning showed that LR model performs best with low regularization. As regParam increased >0.1, accuracy dropped significantly, indicate high penalty values were supressing important feature info.

In [102]:
#Best model in Logistic Regression
import numpy as np
bestLR = LRmodel.bestModel

#Extract coefficient matrix
weights = bestLR.coefficientMatrix.toArray()

#Average absolute importance across 3 species
avgImportance = np.mean(np.abs(weights), axis=0)

#Map each value to its feature name
feature_LR = list(zip(iris.feature_names, avgImportance))
print("Logistic Regression Feature Importance:")
for feature, score in feature_LR:
  print(f"{feature}: {score:.4f}")

Logistic Regression Feature Importance:
sepal length (cm): 0.7799
sepal width (cm): 1.5073
petal length (cm): 0.8324
petal width (cm): 2.0104


In [103]:
#Random Forest
from pyspark.ml.classification import RandomForestClassifier
RF = RandomForestClassifier(featuresCol='features', labelCol='label', seed=42)
paramGrid_RF = (ParamGridBuilder()
              .addGrid(RF.numTrees, [10, 20, 50])
              .addGrid(RF.maxDepth, [5, 10])
              .addGrid(RF.impurity, ['gini', 'entropy'])
              .build())

In [104]:
#Cross-validation and model fitting
cv_RF = CrossValidator(
    estimator=RF,
    estimatorParamMaps=paramGrid_RF,
    evaluator=evaluator,
    numFolds=3,
    parallelism=4
)
RFmodel = cv_RF.fit(train_data)

In [105]:
#Extract parameters and average accuracy scores
RFparameters = RFmodel.getEstimatorParamMaps()
RFmetrics = RFmodel.avgMetrics

#zip together and sort to see the best performing combinations at the top
RFresults = sorted(zip(RFmetrics, RFparameters), key=lambda x: x[0], reverse=True)

print("Random Forest Hyperparameter Ranking (Best to Worst):")
for score, param_map in RFresults:
  params = {p.name: v for p, v in param_map.items()}
  print(f"Accuracy: {score:.4f} | Parameters: {params}")

Random Forest Hyperparameter Ranking (Best to Worst):
Accuracy: 0.9571 | Parameters: {'numTrees': 10, 'maxDepth': 5, 'impurity': 'gini'}
Accuracy: 0.9571 | Parameters: {'numTrees': 10, 'maxDepth': 10, 'impurity': 'gini'}
Accuracy: 0.9475 | Parameters: {'numTrees': 50, 'maxDepth': 5, 'impurity': 'entropy'}
Accuracy: 0.9475 | Parameters: {'numTrees': 50, 'maxDepth': 10, 'impurity': 'gini'}
Accuracy: 0.9475 | Parameters: {'numTrees': 50, 'maxDepth': 10, 'impurity': 'entropy'}
Accuracy: 0.9463 | Parameters: {'numTrees': 20, 'maxDepth': 5, 'impurity': 'gini'}
Accuracy: 0.9380 | Parameters: {'numTrees': 20, 'maxDepth': 5, 'impurity': 'entropy'}
Accuracy: 0.9380 | Parameters: {'numTrees': 20, 'maxDepth': 10, 'impurity': 'entropy'}
Accuracy: 0.9380 | Parameters: {'numTrees': 50, 'maxDepth': 5, 'impurity': 'gini'}
Accuracy: 0.9368 | Parameters: {'numTrees': 20, 'maxDepth': 10, 'impurity': 'gini'}
Accuracy: 0.9261 | Parameters: {'numTrees': 10, 'maxDepth': 5, 'impurity': 'entropy'}
Accuracy: 0.9

In [106]:
#Best model from crossvalidator
bestRF = RFmodel.bestModel

#Feature importance scores
importance = bestRF.featureImportances

feature_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
feature_importance = list(zip(feature_names, importance))
print("Random Forest Feature Importance:")
for feature, score in feature_importance:
  print(f"{feature}: {score:.4f}")

Random Forest Feature Importance:
sepal_length: 0.0739
sepal_width: 0.0045
petal_length: 0.4529
petal_width: 0.4687


Petal width combining with petal_length, demonstrated approximately 90% importance (decision-making power/primary drivers of the classification), showed that barely needs to look at sepals to identify species.

Petals may be much better indicators of species than the sepals.

In [107]:
#Get parameters used in the best model
print(f"Best Max Depth: {bestRF.getMaxDepth()}")
print(f"Best Number of Trees: {bestRF.getOrDefault('numTrees')}")
print(f"Best Impurity: {bestRF.getImpurity()}")

Best Max Depth: 5
Best Number of Trees: 10
Best Impurity: gini


In [108]:
#Decision Tree
from pyspark.ml.classification import DecisionTreeClassifier
DT = DecisionTreeClassifier(
    featuresCol='features',
    labelCol='label',
    seed=42
)
paramGrid_DT = (ParamGridBuilder()
                .addGrid(DT.maxDepth, [2, 5, 10])
                .addGrid(DT.maxBins, [20, 30])
                .addGrid(DT.impurity, ['gini', 'entropy'])
                .build())

In [109]:
#Cross Validation and training
cv_DT = CrossValidator(
    estimator=DT,
    estimatorParamMaps=paramGrid_DT,
    evaluator=evaluator,
    numFolds=3
)
DTmodel = cv_DT.fit(train_data)

In [110]:
#Extract parameters and average accuracy scores
DTparameters = DTmodel.getEstimatorParamMaps()
DTmetrics = DTmodel.avgMetrics

#zip together and sort to see the best performing combinations at the top
DTresults = sorted(zip(DTmetrics, DTparameters), key=lambda x: x[0], reverse=True)

print("Decision Tree Hyperparameter Ranking (Best to Worst):")
for score, param_map in DTresults:
  params = {p.name: v for p, v in param_map.items()}
  print(f"Accuracy: {score:.4f} | Parameters: {params}")

Decision Tree Hyperparameter Ranking (Best to Worst):
Accuracy: 0.9392 | Parameters: {'maxDepth': 5, 'maxBins': 20, 'impurity': 'gini'}
Accuracy: 0.9392 | Parameters: {'maxDepth': 5, 'maxBins': 20, 'impurity': 'entropy'}
Accuracy: 0.9392 | Parameters: {'maxDepth': 5, 'maxBins': 30, 'impurity': 'gini'}
Accuracy: 0.9392 | Parameters: {'maxDepth': 5, 'maxBins': 30, 'impurity': 'entropy'}
Accuracy: 0.9380 | Parameters: {'maxDepth': 2, 'maxBins': 20, 'impurity': 'entropy'}
Accuracy: 0.9380 | Parameters: {'maxDepth': 2, 'maxBins': 30, 'impurity': 'entropy'}
Accuracy: 0.9285 | Parameters: {'maxDepth': 10, 'maxBins': 20, 'impurity': 'gini'}
Accuracy: 0.9285 | Parameters: {'maxDepth': 10, 'maxBins': 20, 'impurity': 'entropy'}
Accuracy: 0.9285 | Parameters: {'maxDepth': 10, 'maxBins': 30, 'impurity': 'gini'}
Accuracy: 0.9285 | Parameters: {'maxDepth': 10, 'maxBins': 30, 'impurity': 'entropy'}
Accuracy: 0.9261 | Parameters: {'maxDepth': 2, 'maxBins': 20, 'impurity': 'gini'}
Accuracy: 0.9261 | Par

Moving from maxDEpth 5 to 10 did not improve accuracy, suggest that rules needed to identify an Iris flower are discovered by the 5th level of the tree.

smaller binning of 20 performed better than 30, indicate splitting the values into too many buckets may hurt model's ability to generalise for small dataset like Iris.

In [111]:
#Feature Importance for Decision Tree
bestDT = DTmodel.bestModel

importance_DT = bestDT.featureImportances
feature_DT = list(zip(feature_names, importance_DT))
print("Decision Tree Feature Importance:")
for feature, score in feature_DT:
  print(f"{feature}: {score:.4f}")

Decision Tree Feature Importance:
sepal_length: 0.0165
sepal_width: 0.0000
petal_length: 0.4381
petal_width: 0.5453


In [112]:
#Best parameter for Decision Tree
print(f"Best Max Depth: {bestDT.getMaxDepth()}")
print(f"Best Max Bins: {bestDT.getMaxBins()}")
print(f"Best Impurity: {bestDT.getImpurity()}")

Best Max Depth: 5
Best Max Bins: 20
Best Impurity: gini


Decision Tree found sepal width to be statistically insignificant for classification.

Similar to RF, petals dominate but with petal length taking lead over petal_width.

Matching of max depth and Gini impurity between DT and RF results, shows consistent models and stable data.

In [113]:
#Predictions and metric calculation for each model
import pandas as pd

#List to hold results
finalComparison = []

models = [
    ('Logistic Regression', LRmodel),
    ('Random Forest', RFmodel),
    ('Decision Tree', DTmodel)
]

for name, fitted_model in models:
  prediction = fitted_model.transform(test_data)

  #Evaluation metrics calculation
  row = {"Model": name}
  for metric in ['accuracy', 'weightedPrecision', 'weightedRecall', 'f1']:
    evaluator = MulticlassClassificationEvaluator(labelCol='label', metricName=metric)
    row[metric] = round(evaluator.evaluate(prediction), 4)

  finalComparison.append(row)

In [114]:
#Evaluation metrics table
df_EM = pd.DataFrame(finalComparison)
df_EM.columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']
df_EM

,Model,Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,0.9643,0.9694,0.9643,0.9647
1,Random Forest,0.9821,0.9835,0.9821,0.9823
2,Decision Tree,0.9821,0.9835,0.9821,0.9823
